# Use k-Means to Cluster Foods Based on Nutrients

In [ ]:
!pip install kmeans-pytorch

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import StandardScaler



pd.options.display.max_rows = 200
pd.options.display.min_rows = 30
pd.options.display.max_seq_items = None
np.random.seed(222)

In [ ]:
# link drive
from google.colab import drive
drive.mount('/content/drive')
share_path = "/content/drive/MyDrive/Colab_Notebooks/deeplearning2026"
import os
print(os.listdir(share_path))

Mounted at /content/drive
['documentation', 'notebooks', 'NLP-Arch.gdoc', 'Siavash', 'data']


In [ ]:
data_path = share_path + '/data/'
df = pd.read_parquet(data_path + 'FDCkmeansclustered.parquet.gzip')
df.columns

Index(['description', 'ALA_G', 'ARA_G', 'B12_UG', 'B1_MG', 'B2_MG', 'B3_MG',
       'B5_MG', 'B6_MG', 'B7_UG', 'B9_UG', 'B_UG', 'CHO1_G', 'CHOfiber1_G',
       'CHOstarch_G', 'CHOsugar_G', 'Ca_MG', 'Cl_MG', 'Co_UG', 'Cr_UG',
       'Cu_MG', 'DHA_G', 'DPA_G', 'EPA_G', 'F_UG', 'Fe_MG', 'I_UG', 'K_MG',
       'MUFA_G', 'Mg_MG', 'Mn_MG', 'Mo_UG', 'Na_MG', 'Ni_UG', 'PUFA_G', 'P_MG',
       'S_MG', 'Se_UG', 'Zn_MG', 'alanine_G', 'alcohol_G', 'arginine_G',
       'aspartate_G', 'caffeine_MG', 'category', 'cholesterol_MG',
       'cysteine_G', 'cystine_G', 'data_type', 'fructose_G', 'glucose_G',
       'glutamate_G', 'glycine_G', 'histidine_G', 'hydroxyproline_G',
       'insol_fiber_G', 'isoleucine_G', 'kcal1_KCAL', 'lactose_G', 'leucine_G',
       'lycopene_UG', 'lysine_G', 'maltose_G', 'methionine_G',
       'phenylalanine_G', 'proline_G', 'protein_G', 'pufa_18_2_n6_G',
       'pufa_18_3_n6_G', 'pufa_20_2_n6_G', 'pufa_20_3_n3_G', 'pufa_20_3_n6_G',
       'pufa_20_4_G', 'sat_fat_G', 'serine_

## Remove excessive missing values

In [ ]:
df.notna().sum(axis=1).describe()

,0
count,487227.000000
mean,25.630012
std,6.897991
min,12.000000
25%,25.000000
50%,25.000000
75%,26.000000
max,88.000000


In [ ]:
# a lot of missing values
# remove observations with fewer than 'required' non-missing values
print(df.shape[0])
required = 25  # will remove ~25% of sample
X = df.loc[df.notna().sum(axis=1) >= required,]
print(X.shape[0])

487227
372592


In [ ]:
# now look at columns with a lot of missing values
n = (X.isna().sum() * 100 / X.shape[0]).sort_values()
print(n)

description          0.000000
k24                  0.000000
k27                  0.000000
k30                  0.000000
text                 0.000000
embeddings           0.000000
k15                  0.000000
k30names             0.000000
k21                  0.000000
k18                  0.000000
data_type            0.027913
total_fat_G          0.053141
category             0.104404
protein_G            0.119702
Na_MG                0.311869
CHO1_G               0.334414
Ca_MG                0.369573
Fe_MG                0.388092
CHOfiber1_G          0.571134
sat_fat_G            0.611393
kcal1_KCAL           0.662924
cholesterol_MG       0.677953
trans_fat_G          3.182570
K_MG                39.020967
vit_c_MG            39.530103
vit_a_IU            44.086561
total_vit_d_IU      57.888521
MUFA_G              80.137791
PUFA_G              80.163020
total_vit_d_UG      80.495555
B3_MG               89.292577
B1_MG               89.431335
B2_MG               89.575729
B6_MG     

In [ ]:
# data frame with less than 90% missing in each column
X = X.drop(n.index[n >= 90], axis = 1)
print(X.shape)


(372592, 33)


## Scale data and remove labels

In [ ]:
X_labels = X[['description','data_type','category']].copy()
X.drop(['description','data_type','category','k24','k27',
        'k30','text','embeddings','k15','k30names','k21','k18'],
       axis = 1, inplace = True)
print(X)
scaler = StandardScaler()
X = scaler.fit_transform(X)


        B1_MG  B2_MG  B3_MG  CHO1_G  CHOfiber1_G  Ca_MG  Fe_MG   K_MG  MUFA_G  \
1         NaN    NaN    NaN    8.24          0.0    0.0   0.85    NaN     NaN   
2         NaN    NaN    NaN   78.26          2.2   22.0   3.04   87.0     NaN   
3         NaN    NaN    NaN    3.33          0.4   42.0   0.15   75.0   0.620   
4         NaN    NaN    NaN    3.33          0.4   42.0   0.15   75.0   0.620   
5         NaN    NaN    NaN   25.47          4.7  123.0   0.17  170.0     NaN   
6         NaN    NaN    NaN   25.47          4.7  123.0   0.17  170.0     NaN   
7       1.000  0.357  8.929   73.21          5.4   21.0   3.57  211.0     NaN   
8         NaN    NaN    NaN    0.00          0.0  214.0   0.00    0.0     NaN   
9         NaN    NaN    NaN   43.24          0.0    0.0   0.00   27.0   0.000   
10        NaN    NaN    NaN   43.24          0.0    0.0   0.00   27.0   0.000   
11        NaN    NaN    NaN   58.82          7.1  188.0   2.54    NaN     NaN   
13        NaN    NaN    NaN 

In [ ]:
k_range = range(15,31,3)

## some functions

In [ ]:
class kmeans_missing(object):
    def __init__(self,potential_centroids,n_clusters):
        #initialize with potential centroids
        self.n_clusters=n_clusters
        self.potential_centroids=potential_centroids
    def fit(self,data,max_iter=10,number_of_runs=1):
        n_clusters=self.n_clusters
        potential_centroids=self.potential_centroids

        dist_mat=np.zeros((data.shape[0],n_clusters))
        all_centroids=np.zeros((n_clusters,data.shape[1],number_of_runs))

        costs=np.zeros((number_of_runs,))
        for k in range(number_of_runs):
            idx=np.random.choice(range(potential_centroids.shape[0]),
                                 size=(n_clusters), replace=False)
            centroids=potential_centroids[idx]
            clusters=np.zeros(data.shape[0])
            old_clusters=np.zeros(data.shape[0])
            for i in range(max_iter):
                #Calc dist to centroids
                for j in range(n_clusters):
                    dist_mat[:,j]=np.nansum((data-centroids[j])**2,axis=1)
                #Assign to clusters
                clusters=np.argmin(dist_mat,axis=1)
                #Update clusters
                for j in range(n_clusters):
                    centroids[j]=np.nanmean(data[clusters==j],axis=0)
                if all(np.equal(clusters,old_clusters)):
                    break # Break when to change in clusters
                if i==max_iter-1:
                    print('no convergence before maximum iterations reached')
                else:
                    clusters,old_clusters=old_clusters,clusters

            all_centroids[:,:,k]=centroids
            costs[k]=np.mean(np.min(dist_mat,axis=1))
        self.costs=costs
        self.cost=np.min(costs)
        self.best_model=np.argmin(costs)
        self.centroids=all_centroids[:,:,self.best_model]
        self.all_centroids=all_centroids
    def predict(self,data):
        dist_mat=np.zeros((data.shape[0],self.n_clusters))
        for j in range(self.n_clusters):
            dist_mat[:,j]=np.nansum((data-self.centroids[j])**2,axis=1)
        prediction=np.argmin(dist_mat,axis=1)
        cost=np.min(dist_mat,axis=1)
        return prediction,cost

## Cluster!

In [ ]:
for k in k_range:
    kmm = kmeans_missing(X, k)
    kmm.fit(X, max_iter=200, number_of_runs=10)
    prediction, cost = kmm.predict(X)
    if len(prediction) != X_labels.shape[0]:
        print(f'kmm{k} has too many rows.')
        break
    X_labels = pd.concat([X_labels,
                      pd.Series(prediction,
                                name = f'kmm{k}',
                                index = X_labels.index)],
                     axis = 1)
    print(f'kmm{k} done')

/tmp/ipykernel_1803/4254212808.py:28: RuntimeWarning: Mean of empty slice
  centroids[j]=np.nanmean(data[clusters==j],axis=0)


kmm15 done


/tmp/ipykernel_1803/4254212808.py:28: RuntimeWarning: Mean of empty slice
  centroids[j]=np.nanmean(data[clusters==j],axis=0)


kmm18 done


/tmp/ipykernel_1803/4254212808.py:28: RuntimeWarning: Mean of empty slice
  centroids[j]=np.nanmean(data[clusters==j],axis=0)


...

no convergence before maximum iterations reached
kmm24 done


/tmp/ipykernel_1803/4254212808.py:28: RuntimeWarning: Mean of empty slice
  centroids[j]=np.nanmean(data[clusters==j],axis=0)


no convergence before maximum iterations reached
kmm27 done


/tmp/ipykernel_1803/4254212808.py:28: RuntimeWarning: Mean of empty slice
  centroids[j]=np.nanmean(data[clusters==j],axis=0)


no convergence before maximum iterations reached
no convergence before maximum iterations reached
kmm30 done


In [ ]:
X_labels


,description,data_type,category,kmm15,kmm18,kmm21,kmm24,kmm27,kmm30
1,all natural gluten free chicken nuggets,branded_food,"Frozen Poultry, Chicken & Turkey",3,13,1,0,15,3
2,all natural rosemary & olive oil basmati rice...,branded_food,Flavored Rice Dishes,14,4,19,20,7,15
3,almond milk,branded_food,Plant Based Milk,8,5,12,23,5,8
4,"almond milk, original",branded_food,Plant Based Milk,8,5,12,23,5,8
5,artisan smooth & creamy gelato,branded_food,Ice Cream & Frozen Yogurt,2,7,0,8,4,18
6,"artisan smooth & creamy gelato, toasted coconut",branded_food,Ice Cream & Frozen Yogurt,2,7,0,8,4,18
7,artisanal collection spaghetti pasta,branded_food,Pasta by Shape & Type,13,4,2,19,22,24
8,authentic barrel ripened feta cheese,branded_food,Cheese,7,9,17,2,20,9
9,barbecue sauce,branded_food,"Ketchup, Mustard, BBQ & Cheese Sauce",8,5,12,23,26,22
10,"barbecue sauce, honey",branded_food,"Ketchup, Mustard, BBQ & Cheese Sauce",8,5,12,23,26,22


In [ ]:
df = pd.concat([X_labels,
                df],
               axis = 1)
df.shape

(487227, 110)

In [ ]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
df

,description,data_type,category,kmm15,kmm18,kmm21,kmm24,kmm27,kmm30,ALA_G,...,water_G,text,embeddings,k15,k18,k21,k24,k27,k30,k30names
1,all natural gluten free chicken nuggets,branded_food,"Frozen Poultry, Chicken & Turkey",3.0,13.0,1.0,0.0,15.0,3.0,NaN,...,NaN,all natural gluten free chicken nuggets,"[-0.038946997, -0.043313783, -0.05184707, -0.0...",3,10,10,1,8,18,poultry
2,all natural rosemary & olive oil basmati rice...,branded_food,Flavored Rice Dishes,14.0,4.0,19.0,20.0,7.0,15.0,NaN,...,NaN,"all natural rosemary olive oil basmati rice, ...","[-0.021323394, -0.0637247, 0.025973855, -0.009...",12,6,0,15,5,25,spice_herb_oil_garlic
3,almond milk,branded_food,Plant Based Milk,8.0,5.0,12.0,23.0,5.0,8.0,NaN,...,NaN,almond milk,"[-0.023772994, -0.03606859, -0.037896283, 0.08...",4,7,14,3,1,27,contains_nuts
4,"almond milk, original",branded_food,Plant Based Milk,8.0,5.0,12.0,23.0,5.0,8.0,NaN,...,NaN,"almond milk, original","[-0.07470463, 0.0006233451, -0.02409026, 0.072...",4,7,14,3,1,27,contains_nuts
5,artisan smooth & creamy gelato,branded_food,Ice Cream & Frozen Yogurt,2.0,7.0,0.0,8.0,4.0,18.0,NaN,...,NaN,artisan smooth creamy gelato,"[-0.02773605, -0.07055933, -0.024659187, 0.044...",12,17,7,0,13,22,cake_pie_waffles
6,"artisan smooth & creamy gelato, toasted coconut",branded_food,Ice Cream & Frozen Yogurt,2.0,7.0,0.0,8.0,4.0,18.0,NaN,...,NaN,"artisan smooth creamy gelato, toasted coconut","[0.013763643, -0.051946856, -0.030013578, 0.08...",6,17,7,0,13,22,cake_pie_waffles
7,artisanal collection spaghetti pasta,branded_food,Pasta by Shape & Type,13.0,4.0,2.0,19.0,22.0,24.0,NaN,...,NaN,artisanal collection spaghetti pasta,"[-0.1018854, 0.04466742, -0.041125428, 0.05308...",13,3,15,2,20,21,pastas_sauces
8,authentic barrel ripened feta cheese,branded_food,Cheese,7.0,9.0,17.0,2.0,20.0,9.0,NaN,...,NaN,authentic barrel ripened feta cheese,"[-0.027281364, 0.03043114, -0.082018316, 0.055...",0,9,20,14,17,5,cheese_in_name
9,barbecue sauce,branded_food,"Ketchup, Mustard, BBQ & Cheese Sauce",8.0,5.0,12.0,23.0,26.0,22.0,NaN,...,NaN,barbecue sauce,"[-0.066953, 0.045108773, -0.0852268, 0.0777444...",10,0,19,20,19,6,hot_bbq_ethnic_sauce
10,"barbecue sauce, honey",branded_food,"Ketchup, Mustard, BBQ & Cheese Sauce",8.0,5.0,12.0,23.0,26.0,22.0,NaN,...,NaN,"barbecue sauce, honey","[-0.05381466, 0.02960883, -0.034579657, 0.0667...",10,0,19,20,19,6,hot_bbq_ethnic_sauce


In [ ]:
df.to_parquet(data_path + 'FDCallclustered.parquet.gzip',compression = 'gzip')